# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/16 22:50:03 WARN Utils: Your hostname, Ulises-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.159 instead (on interface en0)
26/09/16 22:50:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/16 22:50:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows


In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

## Step 1: Keep only the columns we need for the join

For the join we only need each patent's number and state (`POSTATE`). Foreign patents have no state: it can show up as `null` or as an empty string (the CSV file uses `""`), so we drop both. Patents without a state can never produce a same-state citation.

In [8]:
patentStates = patents.select(col("PATENT"), col("POSTATE").alias("STATE")) \
    .filter(col("STATE").isNotNull() & (col("STATE") != ""))
patentStates.show(5)

+-------+-----+
| PATENT|STATE|
+-------+-----+
|3070802|   TX|
|3070803|   IL|
|3070804|   OH|
|3070805|   CA|
|3070806|   PA|
+-------+-----+
only showing top 5 rows


## Step 2: Look up the state of the *cited* patent

Join the citations with `patentStates` on `CITED == PATENT`. An inner join drops citations whose cited patent has no state or isn't in the patent table, which is fine since those citations can't count.

In [9]:
citedStates = citations.join(patentStates, citations.CITED == patentStates.PATENT, "inner") \
    .select(col("CITED"), col("STATE").alias("CITED_STATE"), col("CITING"))
citedStates.show(5)

+-------+-----------+-------+
|  CITED|CITED_STATE| CITING|
+-------+-----------+-------+
|3070803|         IL|4133055|
|3070803|         IL|4253313|
|3070803|         IL|4483021|
|3070803|         IL|4484363|
|3070803|         IL|4921141|
+-------+-----------+-------+
only showing top 5 rows


## Step 3: Look up the state of the *citing* patent

Join again, this time on `CITING == PATENT`. The result is the intermediate table from the README: `CITED, CITED_STATE, CITING, CITING_STATE`. We cache it because it's the expensive part and we use it again.

In [10]:
bothStates = citedStates.join(patentStates, citedStates.CITING == patentStates.PATENT, "inner") \
    .select(col("CITED"), col("CITED_STATE"), col("CITING"), col("STATE").alias("CITING_STATE")) \
    .cache()
bothStates.show(5)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|3815160|         NY|3858597|          MT|
|3675252|         AZ|3858597|          MT|
|3741706|         OH|3859029|          NY|
|3685936|         OH|3859029|          NY|
|3368197|         MI|3859627|          MI|
+-------+-----------+-------+------------+
only showing top 5 rows


## Step 4: Keep same-state citations and count them per citing patent

Filter to rows where the two states match, then group by the citing patent and count.

In [11]:
sameStateCounts = bothStates.filter(col("CITED_STATE") == col("CITING_STATE")) \
    .groupBy("CITING") \
    .agg(count("CITED").alias("SAME_STATE"))
sameStateCounts.show(5)

+-------+----------+
| CITING|SAME_STATE|
+-------+----------+
|3859627|         1|
|3860191|         1|
|3861180|         1|
|3861473|         2|
|3862577|         1|
+-------+----------+
only showing top 5 rows


## Step 5: Add the count to the original patent data

Left-join the counts back onto the full `patents` table, so every patent keeps all its original columns and gets a new `SAME_STATE` column. Patents with no same-state citations get `null`, which we replace with 0 so that sorting works cleanly.

In [12]:
augmented = patents.join(sameStateCounts, patents.PATENT == sameStateCounts.CITING, "left") \
    .drop("CITING") \
    .fillna(0, subset=["SAME_STATE"])

## Step 6: Show the top 10 patents by same-state citations

Sort by `SAME_STATE` in descending order. `PATENT` breaks ties so the output is deterministic.

In [13]:
augmented.orderBy(col("SAME_STATE").desc(), col("PATENT").desc()).show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|       103|
|6008204| 1999|14606|   1998| 